# 01 - Extracao incremental -> Bronze

Busca issues e PRs do GitHub **a partir do watermark de cada repositorio** e grava o JSON cru no Bronze (append-only).

Decisoes de desenho:
- A API e consultada com `since=<watermark>`, `sort=updated`, `direction=asc`.
- O watermark novo e o maior `updated_at` **recebido** (nunca o relogio local) e so avanca **depois** do commit no Bronze.
- Ordem ascendente + limite de paginas = extracao retomavel: se truncar, a proxima execucao continua de onde parou.
- `since` e inclusivo, entao o ultimo item volta a ser buscado. Isso gera duplicata no Bronze, resolvida pelo MERGE no Silver.

In [0]:
%run ./00_common

In [0]:
import requests
import datetime

dbutils.widgets.text("repos", "delta-io/delta,dbt-labs/dbt-core,great-expectations/great_expectations")
dbutils.widgets.text("backfill_days", "7")
dbutils.widgets.text("max_pages_per_repo", "10")
dbutils.widgets.dropdown("full_refresh", "false", ["false", "true"])

REPOS = [r.strip() for r in dbutils.widgets.get("repos").split(",") if r.strip()]
BACKFILL_DAYS = int(dbutils.widgets.get("backfill_days"))
MAX_PAGES = int(dbutils.widgets.get("max_pages_per_repo"))
FULL_REFRESH = dbutils.widgets.get("full_refresh") == "true"

# Token e opcional. Sem token: 60 req/h. Com token: 5000 req/h.
# Para usar: crie um secret scope "github" com a chave "token".
try:
    GITHUB_TOKEN = dbutils.secrets.get("github", "token")
except Exception:
    GITHUB_TOKEN = None

print(f"Repos: {REPOS}")
print(f"Backfill (primeira carga): {BACKFILL_DAYS} dias | max paginas/repo: {MAX_PAGES} | full_refresh: {FULL_REFRESH}")
print(f"Autenticado: {GITHUB_TOKEN is not None}")

In [0]:
from datetime import datetime, timedelta, timezone
import time
import json
from pyspark.sql.types import StructType, StructField, StringType, LongType
import pyspark.sql.functions as F

API = "https://api.github.com"

class GitHubRateLimit(Exception):
    pass

def _get(url, headers, params=None, retries=3):
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, headers=headers, params=params, timeout=30)
        except requests.RequestException:
            if attempt == retries:
                raise
            time.sleep(2 ** attempt)
            continue

        if r.status_code in (403, 429):
            if r.headers.get("X-RateLimit-Remaining") == "0":
                reset = int(r.headers.get("X-RateLimit-Reset", "0"))
                reset_at = datetime.fromtimestamp(reset, timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
                raise GitHubRateLimit(f"rate limit esgotado; reseta as {reset_at} UTC")
            if r.headers.get("Retry-After") and attempt < retries:
                time.sleep(min(int(r.headers["Retry-After"]), 60))
                continue

        if r.status_code >= 500 and attempt < retries:
            time.sleep(2 ** attempt)
            continue

        r.raise_for_status()
        return r
    raise RuntimeError("falha apos retries")

def fetch_issues(repo, since_iso, token=None, max_pages=10, per_page=100):
    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
        "User-Agent": "incremental-pipeline-portfolio",
    }
    if token:
        headers["Authorization"] = f"Bearer {token}"

    url = f"{API}/repos/{repo}/issues"
    params = {"state": "all", "since": since_iso, "sort": "updated",
              "direction": "asc", "per_page": per_page}
    items, pages, remaining = [], 0, None

    while url and pages < max_pages:
        r = _get(url, headers, params)
        params = None
        items.extend(r.json())
        pages += 1
        remaining = r.headers.get("X-RateLimit-Remaining")
        url = r.links.get("next", {}).get("url")

    return items, url is not None, pages, remaining

BRONZE_INPUT_SCHEMA = StructType([
    StructField("repo", StringType()),
    StructField("issue_id", LongType()),
    StructField("updated_at", StringType()),
    StructField("payload", StringType()),
    StructField("run_id", StringType()),
])

In [0]:
# new_run_id, now_str, get_watermark, to_iso, from_iso, set_watermark, log_run, BRONZE are assumed to be defined in ./00_common

run_id = new_run_id()
run_ts = now_str()
default_since = (datetime.now(timezone.utc) - timedelta(days=BACKFILL_DAYS)).strftime("%Y-%m-%d %H:%M:%S")

results = []
failed = False

for repo in REPOS:
    entity = f"extract:{repo}"
    started = now_str()
    try:
        wm, _ = get_watermark(entity)
        since = default_since if (FULL_REFRESH or wm is None) else wm

        items, truncated, calls, remaining = fetch_issues(
            repo, to_iso(since), GITHUB_TOKEN, MAX_PAGES
        )

        if items:
            rows = [(repo, it["id"], it["updated_at"], json.dumps(it), run_id) for it in items]
            df = spark.createDataFrame(rows, BRONZE_INPUT_SCHEMA).select(
                "*", F.lit(run_ts).cast("timestamp").alias("ingested_at")
            )
            df.write.format("delta").mode("append").saveAsTable(BRONZE)
            new_wm = from_iso(max(it["updated_at"] for it in items))
        else:
            new_wm = wm or since

        set_watermark(entity, new_wm, run_id, coverage_str=since)

        detail = f"chamadas={calls}; rate_remaining={remaining}; truncado={truncated}; since={since}"
        log_run(run_id, "extract", entity, started, "SUCCESS", rows_in=len(items), detail=detail)
        results.append((repo, "SUCCESS", len(items), truncated, new_wm))

    except GitHubRateLimit as e:
        failed = True
        log_run(run_id, "extract", entity, started, "FAILED", detail=str(e))
        results.append((repo, "RATE_LIMIT", 0, None, None))
        break  # o limite e compartilhado entre repositorios
    except Exception as e:
        failed = True
        log_run(run_id, "extract", entity, started, "FAILED", detail=f"{type(e).__name__}: {e}")
        results.append((repo, "FAILED", 0, None, None))




In [0]:

summary = spark.createDataFrame(
    results, "repo STRING, status STRING, itens_buscados INT, truncado BOOLEAN, novo_watermark STRING"
)
display(summary)

print(f"run_id: {run_id}")
print("Linhas no Bronze:", spark.table(BRONZE).count())

if failed:
    # Falha visivel: um Job do Databricks marcaria esta execucao como falha.
    raise Exception("Extracao incompleta: veja etl_run_log (status FAILED / RATE_LIMIT).")